# Phase 1 — MedQA Multi-Turn Uncertainty Experiment

Three-round clarifying-question pipeline on a mixed-difficulty MedQA held-out set.

**Dataset:** `multiturn_100.jsonl` — 50 easy / 30 medium / 20 hard, no overlap with `unseen_100.jsonl`

**Pipeline per record:**
1. Model sees `ehr_summary` + `question` + `options` → preliminary A/B/C/D + CQ1 + confidence
2. Patient simulator answers CQ1 → model gives updated A/B/C/D + CQ2 + confidence
3. Patient simulator answers CQ2 → model gives updated A/B/C/D + CQ3 + confidence
4. Patient simulator answers CQ3 → model gives final A/B/C/D + final confidence

**Output:** one row per case with assessments at all 4 checkpoints (prelim + after each CQ)

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../../").resolve()))

from config import SIMULATOR_MODEL_ID  # single source of truth — edit config.py to change

# ── Dataset config ────────────────────────────────────────────────────────
DATASET               = "medqa"
ROOT                  = Path("../../").resolve()
PROMPTS_DIR           = ROOT / "prompts"  / DATASET
DATASETS_DIR          = ROOT / "datasets" / DATASET

MODEL_ID              = "gemini-2.5-flash"  # clinician model
OUTPUTS_DIR           = ROOT / "outputs" / DATASET / MODEL_ID

CASES_PATH            = DATASETS_DIR / "multiturn_100.jsonl"
INSTRUCTION_FILE      = PROMPTS_DIR  / "phase1_instruction.txt"
CONTINUATION_FILE     = PROMPTS_DIR  / "phase1_continuation_instruction.txt"
OUTPUT_CSV            = OUTPUTS_DIR  / "phase1_multiturn_results.csv"

# ── Run config ────────────────────────────────────────────────────────────
N_CQ_TURNS            = 3
REQUEST_INTERVAL      = 1.0
# ─────────────────────────────────────────────────────────────────────────

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"ROOT:         {ROOT}")
print(f"Cases:        {CASES_PATH}")
print(f"Instruction:  {INSTRUCTION_FILE}")
print(f"Continuation: {CONTINUATION_FILE}")
print(f"Output CSV:   {OUTPUT_CSV}")
print(f"Clinician:    {MODEL_ID}")
print(f"Simulator:    {SIMULATOR_MODEL_ID}")

In [2]:
import importlib
import json
import logging

import pandas as pd
from IPython.display import display, Markdown

import src.utils as utils_mod
import src.providers as providers_mod
import src.pipeline as pipeline_mod
importlib.reload(utils_mod)
importlib.reload(providers_mod)
importlib.reload(pipeline_mod)

from src.utils import load_dotenv, parse_json_response, format_answer_choices
from src.providers import GeminiProvider
from src.pipeline import (
    MultiTurnPhase1Pipeline, PatientSimulator,
    TURN_0_SCHEMA, TURN_CONTINUATION_SCHEMA, TURN_1_SCHEMA,
    _POST_CLARIFICATION_INSTRUCTION,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%H:%M:%S",
)

load_dotenv(ROOT / ".env")
print("Environment loaded. src modules reloaded.")

Environment loaded. src modules reloaded.


## Smoke Test

In [3]:
provider_smoke = GeminiProvider(model_id=MODEL_ID)
response = provider_smoke.call(
    system_instruction="You are a test assistant.",
    user_message="Reply with exactly: SMOKE TEST PASSED",
    temperature=0.0,
    max_tokens=512,
)
assert len(response.strip()) > 0, "Smoke test failed — empty response"
print(f"Smoke test response: {response.strip()}")
print("Smoke test passed.")

01:14:48 [INFO] src.providers — GeminiProvider ready — model=gemini-2.5-flash api_version=v1beta


01:14:48 [INFO] google_genai.models — AFC is enabled with max remote calls: 10.


01:14:51 [INFO] httpx — HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


Smoke test response: SMOKE TEST PASSED
Smoke test passed.


## Load Dataset

In [4]:
from src.utils import clean_simulator_context

with open(CASES_PATH, encoding="utf-8") as f:
    raw_cases = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(raw_cases)} cases from {CASES_PATH.name}")

records = []
for c in raw_cases:
    simulator_context = "\n\n---\n\n".join(
        ctx for ctx in [
            clean_simulator_context(c.get("patient_context", "")),
            clean_simulator_context(c.get("nurse_context", "")),
            clean_simulator_context(c.get("specialist_context", "")),
        ] if ctx and ctx.strip()
    )
    records.append({
        "id":               c["case_id"],
        "ehr_summary":      c["ehr_summary"],
        "question":         c["question"],
        "options":          c["options"],
        "correct_option":   c["correct_option"],
        "correct_answer":   c["correct_answer"],
        "simulator_context": simulator_context,
        "difficulty":       c.get("difficulty", ""),
    })

print(f"Records prepared: {len(records)}")

# Verify cleaning removed diagnostic content
import re as _re
leaks = sum(1 for r in records if _re.search(
    r"most consistent with|Diagnosis:", r["simulator_context"], _re.IGNORECASE))
print(f"Diagnostic leakage check: {leaks}/{len(records)} contexts still contain conclusion language")

# ── Scientific validity checks ────────────────────────────────────────────
# 1. No overlap with unseen_100.jsonl
with open(DATASETS_DIR / "unseen_100.jsonl", encoding="utf-8") as f:
    old_ids = {json.loads(l)["case_id"] for l in f if l.strip()}
new_ids = {r["id"] for r in records}
overlap = new_ids & old_ids
assert len(overlap) == 0, f"DATA LEAKAGE: {len(overlap)} IDs overlap with unseen_100.jsonl: {overlap}"
print(f"Leakage check PASSED — 0 overlap with unseen_100.jsonl")

# 2. Correct difficulty distribution
from collections import Counter
diff_counts = Counter(r["difficulty"] for r in records)
print(f"Difficulty distribution: {dict(diff_counts)}")
assert diff_counts["easy"] == 50 and diff_counts["medium"] == 30 and diff_counts["hard"] == 20, \
    f"Unexpected difficulty split: {dict(diff_counts)}"
print("Difficulty split check PASSED — 50 easy / 30 medium / 20 hard")


Loaded 100 cases from multiturn_100.jsonl


Records prepared: 100
Diagnostic leakage check: 0/100 contexts still contain conclusion language
Leakage check PASSED — 0 overlap with unseen_100.jsonl
Difficulty distribution: {'easy': 50, 'medium': 30, 'hard': 20}
Difficulty split check PASSED — 50 easy / 30 medium / 20 hard


## Dry Run — Single Record
Verify the full three-turn flow on one record before running everything.

In [ ]:
provider           = GeminiProvider(model_id=MODEL_ID)
simulator_provider = GeminiProvider(model_id=SIMULATOR_MODEL_ID)
simulator          = PatientSimulator(simulator_provider)
instruction      = INSTRUCTION_FILE.read_text(encoding="utf-8").strip()
continuation_ins = CONTINUATION_FILE.read_text(encoding="utf-8").strip()

test = records[0]
print(f"Testing on: {test['id']} | difficulty={test['difficulty']}")
print(f"EHR: {test['ehr_summary']}")
print(f"Q:   {test['question']}")
print(f"Options: {test['options']}")
print(f"Correct: {test['correct_option']} | {test['correct_answer']}")
print()

history = []  # list of (cq, sim_response)

# Turn 0
user_msg_0 = (
    f"Patient presentation:\n{test['ehr_summary'].strip()}\n\n"
    f"Clinical question:\n{test['question'].strip()}\n\n"
    f"Answer options:\n{format_answer_choices(test['options'])}"
)
raw_0 = provider.call(
    system_instruction=instruction,
    user_message=user_msg_0,
    temperature=0.0, max_tokens=4096, expect_json=TURN_0_SCHEMA,
)
p0 = parse_json_response(raw_0)
print(f"=== TURN 0 ===\n{json.dumps(p0, indent=2)}")
history.append((p0["clarifying_question"], None))

# CQ rounds
for turn_idx in range(1, N_CQ_TURNS + 1):
    cq = history[-1][0]
    sim = simulator.answer(cq, test["simulator_context"])
    history[-1] = (cq, sim)
    print(f"\n=== SIM {turn_idx} ===\n{sim}")

    history_text = "\n\n".join(
        f"Clarifying question {i}: {q}\nPatient's answer: {a}"
        for i, (q, a) in enumerate(history, 1)
    )
    base_msg = (
        f"Patient presentation:\n{test['ehr_summary'].strip()}\n\n"
        f"Clinical question:\n{test['question'].strip()}\n\n"
        f"Answer options:\n{format_answer_choices(test['options'])}\n\n"
        f"{history_text}"
    )

    if turn_idx < N_CQ_TURNS:
        raw = provider.call(
            system_instruction=continuation_ins,
            user_message=base_msg,
            temperature=0.0, max_tokens=4096, expect_json=TURN_CONTINUATION_SCHEMA,
        )
        p = parse_json_response(raw)
        print(f"\n=== TURN {turn_idx} (continuation) ===\n{json.dumps(p, indent=2)}")
        history.append((p["clarifying_question"], None))
    else:
        raw = provider.call(
            system_instruction=_POST_CLARIFICATION_INSTRUCTION,
            user_message=base_msg,
            temperature=0.0, max_tokens=4096, expect_json=TURN_1_SCHEMA,
        )
        p = parse_json_response(raw)
        print(f"\n=== TURN {turn_idx} (FINAL) ===\n{json.dumps(p, indent=2)}")

print(f"\nCorrect answer: {test['correct_option']} | {test['correct_answer']}")

## Run Full Experiment
100 cases × 3 CQ rounds = 300 API call groups. Resumes automatically if interrupted.

In [ ]:
provider           = GeminiProvider(model_id=MODEL_ID)
simulator_provider = GeminiProvider(model_id=SIMULATOR_MODEL_ID)

pipeline = MultiTurnPhase1Pipeline(
    provider=provider,
    instruction_file=INSTRUCTION_FILE,
    continuation_instruction_file=CONTINUATION_FILE,
    output_csv=OUTPUT_CSV,
    n_turns=N_CQ_TURNS,
    request_interval=REQUEST_INTERVAL,
    simulator_provider=simulator_provider,
)

pipeline.run(records)

## Results Summary

In [7]:
results_df = pd.read_csv(OUTPUT_CSV)
valid = results_df[~results_df["was_blocked"]]

print(f"Total records: {len(results_df)} | Blocked: {results_df['was_blocked'].sum()}")
print()

checkpoints = [
    ("Preliminary (Turn 0)",  "preliminary_assessment", "is_correct_preliminary", "preliminary_confidence"),
    ("After CQ1",             "assessment_1",           "is_correct_1",           "confidence_1"),
    ("After CQ2",             "assessment_2",           "is_correct_2",           "confidence_2"),
    ("After CQ3 (Final)",     "final_assessment",       "is_correct_final",       "final_confidence"),
]

print(f"{'Checkpoint':<25} {'Correct':>10} {'Accuracy':>10} {'Mean Conf':>10}")
print("-" * 60)
for label, _, correct_col, conf_col in checkpoints:
    n_correct = valid[correct_col].sum()
    acc = valid[correct_col].mean()
    mean_conf = valid[conf_col].mean()
    print(f"{label:<25} {n_correct:>10} {acc:>10.1%} {mean_conf:>10.1f}")

print()
print("Per-difficulty breakdown (final accuracy):")
display(
    valid.groupby("difficulty")[["is_correct_preliminary", "is_correct_1", "is_correct_2", "is_correct_final"]]
    .mean().round(3)
)

Total records: 100 | Blocked: 0

Checkpoint                   Correct   Accuracy  Mean Conf
------------------------------------------------------------
Preliminary (Turn 0)              57      57.0%       63.0
After CQ1                         69      69.0%       80.6
After CQ2                         72      72.0%       85.9
After CQ3 (Final)                 75      75.0%       89.7

Per-difficulty breakdown (final accuracy):


,is_correct_preliminary,is_correct_1,is_correct_2,is_correct_final
difficulty,,,,
easy,0.680,0.70,0.74,0.760
hard,0.450,0.65,0.70,0.750
medium,0.467,0.70,0.70,0.733
